# Notebook 5 — Rules Mining (Apriori, Proportional Department Selection)

**Goal:** Mine association rules with Apriori using the `train_rules` baskets.

This notebook:
- Loads baskets created in Notebook 4 (from `outputs/`)
- Selects ~3000 products with a **department-proportional strategy** based on product occurrence shares
- Builds a sparse one-hot transaction matrix (memory-safe)
- Mines frequent itemsets with Apriori (`max_len=3`)
- Builds association rules (support, confidence, lift)
- Keeps rules with single-item consequents (subset -> item)
- Saves itemsets/rules and selection diagnostics to `outputs/`

In [34]:
import os
import ast
import pandas as pd
import numpy as np
from IPython.display import display
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [35]:
def load_baskets_outputs():
    """
    Load basket files created in Notebook 4 from the outputs folder.

    Returns
    -------
    dict
        Basket tables (all, train_rules, validation, test).
    """
    tables = {
        "baskets_all": pd.read_csv("../outputs/baskets_all.csv"),
        "baskets_train_rules": pd.read_csv("../outputs/baskets_train_rules.csv"),
        "baskets_validation": pd.read_csv("../outputs/baskets_validation.csv"),
        "baskets_test": pd.read_csv("../outputs/baskets_test.csv"),
    }
    return tables


def load_product_tables():
    """
    Load Instacart product metadata tables.

    Returns
    -------
    tuple
        (products, aisles, departments)
    """
    products = pd.read_csv("../data/products.csv")
    aisles = pd.read_csv("../data/aisles.csv")
    departments = pd.read_csv("../data/departments.csv")
    return products, aisles, departments


def parse_items_column(baskets_df):
    """
    Convert the 'items' column from string to Python list.

    Returns
    -------
    DataFrame
        Same table with parsed 'items' column.
    """
    df = baskets_df.copy()
    df["items"] = df["items"].apply(ast.literal_eval)
    return df


def basket_overview(baskets_df, name):
    """
    Build a quick overview of a basket table.

    Returns
    -------
    DataFrame
        One-row summary.
    """
    row = {
        "table": name,
        "n_baskets": baskets_df["order_id"].nunique() if "order_id" in baskets_df.columns else len(baskets_df),
        "n_users": baskets_df["user_id"].nunique() if "user_id" in baskets_df.columns else np.nan,
        "avg_basket_size": baskets_df["items"].apply(len) if "items" in baskets_df.columns else np.nan,
    }
    # avg/median computed separately to keep function simple for beginners
    if "items" in baskets_df.columns:
        row["avg_basket_size"] = baskets_df["items"].apply(len).mean()
        row["median_basket_size"] = baskets_df["items"].apply(len).median()
    else:
        row["avg_basket_size"] = np.nan
        row["median_basket_size"] = np.nan

    return pd.DataFrame([row])


def enrich_products(products, aisles, departments):
    """
    Merge products with aisles and departments.

    Returns
    -------
    DataFrame
        Product metadata with aisle and department names.
    """
    df = products.merge(aisles, on="aisle_id", how="left")
    df = df.merge(departments, on="department_id", how="left")
    return df


def compute_product_occurrences_with_department(baskets_df, products_enriched):
    """
    Compute product occurrence counts in baskets and attach department metadata.

    Parameters
    ----------
    baskets_df : DataFrame
        Basket table with 'items' column.
    products_enriched : DataFrame
        Product metadata table.

    Returns
    -------
    DataFrame
        Product occurrence table with department/aisle columns.
    """
    exploded = baskets_df[["order_id", "items"]].explode("items").rename(columns={"items": "product_id"})
    occ = exploded.groupby("product_id").size().reset_index(name="n_occurrences")

    occ = occ.merge(
        products_enriched[["product_id", "product_name", "department_id", "department", "aisle_id", "aisle"]],
        on="product_id",
        how="left"
    )

    # Keep only rows with known department
    occ = occ[occ["department"].notna()].copy()

    return occ


def allocate_department_quotas_from_occurrences(product_occ, target_n_products=3000):
    """
    Allocate product quotas per department proportionally to occurrence shares.

    We use:
    - department share = department occurrences / total occurrences
    - quota = target_n_products * share
    - largest remainder method to ensure total quotas sum exactly to target_n_products

    Parameters
    ----------
    product_occ : DataFrame
        Output of compute_product_occurrences_with_department().
    target_n_products : int
        Target number of products to select.

    Returns
    -------
    DataFrame
        Department quotas with shares and final allocated quota.
    """
    dep = product_occ.groupby("department").agg(
        dept_occurrences=("n_occurrences", "sum"),
        n_available_products=("product_id", "nunique")
    ).reset_index()

    total_occ = dep["dept_occurrences"].sum()
    dep["occ_share"] = dep["dept_occurrences"] / total_occ

    # Raw quota before rounding
    dep["quota_raw"] = dep["occ_share"] * target_n_products

    # Floor first
    dep["quota_floor"] = np.floor(dep["quota_raw"]).astype(int)
    dep["quota_remainder"] = dep["quota_raw"] - dep["quota_floor"]

    # Start with floor quota, but cannot exceed available products
    dep["quota"] = dep[["quota_floor", "n_available_products"]].min(axis=1)

    # Distribute remaining quota using largest remainders, respecting availability
    current_total = int(dep["quota"].sum())
    remaining = int(target_n_products - current_total)

    if remaining > 0:
        dep = dep.sort_values("quota_remainder", ascending=False).reset_index(drop=True)

        # Multiple passes in case some departments are full
        while remaining > 0:
            changed = False
            for i in dep.index:
                if dep.loc[i, "quota"] < dep.loc[i, "n_available_products"]:
                    dep.loc[i, "quota"] += 1
                    remaining -= 1
                    changed = True
                    if remaining == 0:
                        break
            if not changed:
                # No more capacity in any department
                break

    # If floor allocation exceeds target because of constraints (rare), trim by smallest remainders
    if dep["quota"].sum() > target_n_products:
        extra = int(dep["quota"].sum() - target_n_products)
        dep = dep.sort_values("quota_remainder", ascending=True).reset_index(drop=True)
        while extra > 0:
            changed = False
            for i in dep.index:
                if dep.loc[i, "quota"] > 0:
                    dep.loc[i, "quota"] -= 1
                    extra -= 1
                    changed = True
                    if extra == 0:
                        break
            if not changed:
                break

    dep = dep.sort_values("dept_occurrences", ascending=False).reset_index(drop=True)

    return dep


def select_products_proportional_by_department(baskets_df, products_enriched, target_n_products=3000):
    """
    Select products proportionally by department, based on occurrence shares.

    Steps
    -----
    1) Compute product occurrences in train_rules baskets
    2) Compute department occurrence shares
    3) Allocate department quotas to sum to target_n_products
    4) Inside each department, keep the most frequent products up to the quota

    Parameters
    ----------
    baskets_df : DataFrame
        Basket table with 'items' column.
    products_enriched : DataFrame
        Product metadata.
    target_n_products : int
        Target number of products to select.

    Returns
    -------
    tuple
        (selected_product_ids, selected_products, department_quota_table)
    """
    product_occ = compute_product_occurrences_with_department(baskets_df, products_enriched)

    # Rank products inside each department by occurrence
    product_occ["rank_in_department"] = product_occ.groupby("department")["n_occurrences"].rank(
        method="first",
        ascending=False
    )

    dep_quota = allocate_department_quotas_from_occurrences(
        product_occ,
        target_n_products=target_n_products
    )

    # Attach quotas to product occurrence table
    product_occ = product_occ.merge(dep_quota[["department", "quota", "occ_share"]], on="department", how="left")

    # Keep top products inside each department according to quota
    selected = product_occ[product_occ["rank_in_department"] <= product_occ["quota"]].copy()

    selected = selected.sort_values(["department", "rank_in_department"]).reset_index(drop=True)

    selected_product_ids = set(selected["product_id"].tolist())

    return selected_product_ids, selected, dep_quota


def build_department_distribution_report_before_after(baskets_df, selected_products, products_enriched):
    """
    Compare department distribution before and after product selection.

    "Before" is based on all product occurrences in baskets_df.
    "After" is based on occurrences of selected products only.

    Returns
    -------
    DataFrame
        Department distribution comparison (before vs after).
    """
    # Before (all occurrences)
    exploded_all = baskets_df[["order_id", "items"]].explode("items").rename(columns={"items": "product_id"})
    exploded_all = exploded_all.merge(
        products_enriched[["product_id", "department"]],
        on="product_id",
        how="left"
    )

    before = exploded_all.groupby("department").size().reset_index(name="occ_before")
    before["share_before"] = before["occ_before"] / before["occ_before"].sum()

    # After (only selected products)
    selected_ids = set(selected_products["product_id"].tolist())

    exploded_after = exploded_all[exploded_all["product_id"].isin(selected_ids)].copy()
    after = exploded_after.groupby("department").size().reset_index(name="occ_after")
    after["share_after"] = after["occ_after"] / after["occ_after"].sum()

    # Number of selected products by department
    selected_counts = selected_products.groupby("department")["product_id"].nunique().reset_index(name="n_selected_products")

    report = before.merge(after, on="department", how="outer")
    report = report.merge(selected_counts, on="department", how="outer")

    report["occ_before"] = report["occ_before"].fillna(0).astype(int)
    report["occ_after"] = report["occ_after"].fillna(0).astype(int)
    report["share_before"] = report["share_before"].fillna(0.0)
    report["share_after"] = report["share_after"].fillna(0.0)
    report["n_selected_products"] = report["n_selected_products"].fillna(0).astype(int)

    report["share_diff_abs"] = (report["share_after"] - report["share_before"]).abs()

    report = report.sort_values("share_before", ascending=False).reset_index(drop=True)

    return report


def prepare_baskets_for_apriori_proportional(
    baskets_df,
    products_enriched,
    target_n_products=3000,
    sample_n_baskets=150000,
    stratified_by_segment=True,
    random_state=42
):
    """
    Prepare baskets for Apriori using proportional department-based product selection.

    Steps
    -----
    1) Select products proportionally by department (based on occurrences)
    2) Filter baskets to selected products
    3) Remove baskets with fewer than 2 items
    4) Sample baskets (optional), stratified by segment if available

    Parameters
    ----------
    baskets_df : DataFrame
        Basket table with 'items' column.
    products_enriched : DataFrame
        Product metadata.
    target_n_products : int
        Target number of selected products.
    sample_n_baskets : int
        Max number of baskets after filtering (None to keep all).
    stratified_by_segment : bool
        If True and segment exists, sample proportionally by segment.
    random_state : int
        Random seed.

    Returns
    -------
    tuple
        (baskets_ready, selected_products, dep_quota, dep_distribution_report, prep_summary)
    """
    df = baskets_df.copy()

    # 1) Proportional product selection
    selected_product_ids, selected_products, dep_quota = select_products_proportional_by_department(
        df,
        products_enriched,
        target_n_products=target_n_products
    )

    dep_distribution_report = build_department_distribution_report_before_after(
        df,
        selected_products,
        products_enriched
    )

    # 2) Filter baskets
    n_baskets_before = len(df)
    df["items"] = df["items"].apply(lambda items: [p for p in items if p in selected_product_ids])

    # 3) Keep baskets with at least 2 items (Apriori co-occurrence)
    df["basket_size_filtered"] = df["items"].apply(len)
    df = df[df["basket_size_filtered"] >= 2].copy()
    n_baskets_after_len_filter = len(df)

    # 4) Optional sampling
    n_before_sample = len(df)

    if sample_n_baskets is not None and len(df) > sample_n_baskets:
        if stratified_by_segment and "segment" in df.columns:
            # Proportional sample by segment
            seg_shares = df["segment"].value_counts(normalize=True)
            parts = []

            for seg, share in seg_shares.items():
                seg_df = df[df["segment"] == seg]
                n_seg = int(round(sample_n_baskets * share))
                n_seg = min(n_seg, len(seg_df))
                if n_seg > 0:
                    parts.append(seg_df.sample(n=n_seg, random_state=random_state))

            sampled = pd.concat(parts, ignore_index=False).drop_duplicates(subset=["order_id"])

            # Fill any missing rows caused by rounding
            missing = sample_n_baskets - len(sampled)
            if missing > 0:
                remaining = df[~df["order_id"].isin(sampled["order_id"])]
                if len(remaining) > 0:
                    extra = remaining.sample(n=min(missing, len(remaining)), random_state=random_state)
                    sampled = pd.concat([sampled, extra], ignore_index=False)

            df = sampled.copy()
        else:
            df = df.sample(n=sample_n_baskets, random_state=random_state).copy()

    n_after_sample = len(df)

    prep_summary = pd.DataFrame([{
        "selection_method": "proportional_by_department_occurrences",
        "target_n_products": target_n_products,
        "n_products_selected": len(selected_product_ids),
        "sample_n_baskets_target": sample_n_baskets if sample_n_baskets is not None else -1,
        "n_baskets_before_filter": n_baskets_before,
        "n_baskets_after_len_filter": n_baskets_after_len_filter,
        "n_baskets_before_sample": n_before_sample,
        "n_baskets_after_sample": n_after_sample,
        "avg_share_diff_abs_departments": dep_distribution_report["share_diff_abs"].mean()
    }])

    return df, selected_products, dep_quota, dep_distribution_report, prep_summary


def build_onehot_matrix(baskets_df):
    """
    Build a sparse one-hot transaction matrix for Apriori.

    Parameters
    ----------
    baskets_df : DataFrame
        Basket table with an 'items' column.

    Returns
    -------
    tuple
        (X_df, col_names)
        - X_df: sparse pandas dataframe (bool)
        - col_names: string product IDs
    """
    te = TransactionEncoder()

    # Sparse matrix to avoid memory issues
    X_sparse = te.fit(baskets_df["items"]).transform(baskets_df["items"], sparse=True)

    # mlxtend + sparse pandas requires string column names
    col_names = [str(c) for c in te.columns_]

    X_df = pd.DataFrame.sparse.from_spmatrix(X_sparse, columns=col_names)

    return X_df, col_names


def sparse_info_from_onehot(X_df):
    """
    Compute sparsity statistics without dense conversion.

    Returns
    -------
    DataFrame
        Matrix size and density info.
    """
    n_rows = X_df.shape[0]
    n_cols = X_df.shape[1]
    total_cells = n_rows * n_cols

    nnz = 0
    for col in X_df.columns:
        nnz += int(X_df[col].sparse.npoints)

    density = nnz / total_cells if total_cells > 0 else 0.0
    sparsity = 1 - density

    return pd.DataFrame([{
        "n_baskets": n_rows,
        "n_products": n_cols,
        "non_zero_values": nnz,
        "total_cells": total_cells,
        "density": density,
        "sparsity": sparsity
    }])


def mine_frequent_itemsets_apriori(X_df, min_support=0.004, max_len=3):
    """
    Mine frequent itemsets using Apriori.

    Parameters
    ----------
    X_df : DataFrame
        Sparse one-hot transaction matrix.
    min_support : float
        Minimum support threshold.
    max_len : int
        Maximum itemset length.

    Returns
    -------
    DataFrame
        Frequent itemsets table.
    """
    itemsets = apriori(
        X_df,
        min_support=min_support,
        use_colnames=True,
        max_len=max_len,
        low_memory=True
    )
    itemsets = itemsets.sort_values("support", ascending=False).reset_index(drop=True)
    return itemsets


def add_itemset_metadata(itemsets):
    """
    Add itemset length metadata.

    Returns
    -------
    DataFrame
        Frequent itemsets with itemset length.
    """
    df = itemsets.copy()
    df["itemset_length"] = df["itemsets"].apply(len)
    return df


def build_association_rules(itemsets, metric="confidence", min_threshold=0.10):
    """
    Build association rules from frequent itemsets.

    Returns
    -------
    DataFrame
        Rules table.
    """
    if len(itemsets) == 0:
        return pd.DataFrame()

    rules = association_rules(itemsets, metric=metric, min_threshold=min_threshold)
    return rules


def add_rule_metadata(rules):
    """
    Add readable metadata columns to the rules table.

    Returns
    -------
    DataFrame
        Rules with metadata columns.
    """
    if len(rules) == 0:
        return rules.copy()

    df = rules.copy()

    df["antecedent_len"] = df["antecedents"].apply(len)
    df["consequent_len"] = df["consequents"].apply(len)
    df["rule_len"] = df["antecedent_len"] + df["consequent_len"]

    df["antecedents_str"] = df["antecedents"].apply(lambda x: ", ".join(sorted(list(x))))
    df["consequents_str"] = df["consequents"].apply(lambda x: ", ".join(sorted(list(x))))

    return df


def keep_single_item_consequents(rules):
    """
    Keep only rules with 1-item consequent (subset -> item).

    Returns
    -------
    DataFrame
        Filtered rules.
    """
    if len(rules) == 0:
        return rules.copy()

    return rules[rules["consequents"].apply(len) == 1].copy()


def sort_rules_for_recommendation(rules):
    """
    Sort rules for recommendation usage.

    Sort priority:
    1) confidence desc
    2) lift desc
    3) support desc
    """
    if len(rules) == 0:
        return rules.copy()

    return rules.sort_values(
        ["confidence", "lift", "support"],
        ascending=[False, False, False]
    ).reset_index(drop=True)


def rules_summary(rules):
    """
    Build a compact summary of the rules table.

    Returns
    -------
    DataFrame
        Rule summary.
    """
    if len(rules) == 0:
        return pd.DataFrame([{
            "n_rules": 0,
            "avg_confidence": np.nan,
            "avg_lift": np.nan,
            "avg_support": np.nan,
            "max_rule_len": np.nan
        }])

    return pd.DataFrame([{
        "n_rules": len(rules),
        "avg_confidence": rules["confidence"].mean(),
        "avg_lift": rules["lift"].mean(),
        "avg_support": rules["support"].mean(),
        "max_rule_len": rules["rule_len"].max()
    }])


def top_itemsets_by_length(itemsets, top_n=10):
    """
    Show top itemsets by support inside each itemset length.

    Returns
    -------
    DataFrame
        Top itemsets grouped by itemset length.
    """
    if len(itemsets) == 0:
        return itemsets.copy()

    tmp = itemsets.copy()
    tmp["rank_in_length"] = tmp.groupby("itemset_length")["support"].rank(method="first", ascending=False)
    tmp = tmp[tmp["rank_in_length"] <= top_n].copy()
    tmp = tmp.sort_values(["itemset_length", "rank_in_length"])
    return tmp


def save_itemsets_and_rules(itemsets, rules, prefix="apriori"):
    """
    Save frequent itemsets and rules to outputs folder.
    """
    os.makedirs("../outputs", exist_ok=True)

    itemsets_to_save = itemsets.copy()
    if "itemsets" in itemsets_to_save.columns:
        itemsets_to_save["itemsets"] = itemsets_to_save["itemsets"].apply(
            lambda x: "|".join(sorted(list(x)))
        )

    rules_to_save = rules.copy()
    if len(rules_to_save) > 0:
        rules_to_save["antecedents"] = rules_to_save["antecedents"].apply(
            lambda x: "|".join(sorted(list(x)))
        )
        rules_to_save["consequents"] = rules_to_save["consequents"].apply(
            lambda x: "|".join(sorted(list(x)))
        )

    itemsets_to_save.to_csv(f"../outputs/{prefix}_frequent_itemsets.csv", index=False)
    rules_to_save.to_csv(f"../outputs/{prefix}_rules.csv", index=False)


def save_selection_outputs(selected_products, dep_quota, dep_distribution_report, prep_summary, prefix="apriori_train_rules"):
    """
    Save product selection diagnostics and preparation summary.
    """
    os.makedirs("../outputs", exist_ok=True)

    selected_products.to_csv(f"../outputs/{prefix}_selected_products.csv", index=False)
    dep_quota.to_csv(f"../outputs/{prefix}_department_quotas.csv", index=False)
    dep_distribution_report.to_csv(f"../outputs/{prefix}_department_distribution_before_after.csv", index=False)
    prep_summary.to_csv(f"../outputs/{prefix}_prep_summary.csv", index=False)

In [36]:
# 1) Load baskets and product metadata
basket_tables = load_baskets_outputs()

baskets_all = parse_items_column(basket_tables["baskets_all"])
baskets_train_rules = parse_items_column(basket_tables["baskets_train_rules"])
baskets_validation = parse_items_column(basket_tables["baskets_validation"])
baskets_test = parse_items_column(basket_tables["baskets_test"])

products, aisles, departments = load_product_tables()
products_enriched = enrich_products(products, aisles, departments)

display(pd.concat([
    basket_overview(baskets_all, "baskets_all"),
    basket_overview(baskets_train_rules, "baskets_train_rules"),
    basket_overview(baskets_validation, "baskets_validation"),
    basket_overview(baskets_test, "baskets_test"),
], ignore_index=True))

,table,n_baskets,n_users,avg_basket_size,median_basket_size
0,baskets_all,3346083,206209,10.107073,8.0
1,baskets_train_rules,3008665,206209,10.069151,8.0
2,baskets_validation,206209,206209,10.376792,9.0
3,baskets_test,131209,131209,10.552759,9.0


In [37]:
# 2) Prepare TRAIN_RULES baskets for Apriori with proportional department selection
# Target: ~3000 products, representative of department occurrence distribution
target_n_products = 3000
sample_n_baskets = 150000

baskets_train_rules_apriori, selected_products_used, dep_quota, dep_distribution_report, prep_summary = prepare_baskets_for_apriori_proportional(
    baskets_train_rules,
    products_enriched,
    target_n_products=target_n_products,
    sample_n_baskets=sample_n_baskets,
    stratified_by_segment=True,
    random_state=42
)

display(prep_summary)

# Department quotas (what we planned)
display(dep_quota.sort_values("quota", ascending=False))

# Compare department distribution before vs after selection (what we achieved)
display(dep_distribution_report)

# Optional segment check after basket sampling
if "segment" in baskets_train_rules_apriori.columns:
    seg_dist = baskets_train_rules_apriori["segment"].value_counts(normalize=True).reset_index()
    seg_dist.columns = ["segment", "pct_baskets_after_sample"]
    display(seg_dist)

# Save preparation diagnostics
save_selection_outputs(
    selected_products_used,
    dep_quota,
    dep_distribution_report,
    prep_summary,
    prefix="apriori_train_rules"
)

,selection_method,target_n_products,n_products_selected,sample_n_baskets_target,n_baskets_before_filter,n_baskets_after_len_filter,n_baskets_before_sample,n_baskets_after_sample,avg_share_diff_abs_departments
0,proportional_by_department_occurrences,3000,3000,150000,3008665,2651892,2651892,150000,0.013037


,department,dept_occurrences,n_available_products,occ_share,quota_raw,quota_floor,quota_remainder,quota
0,produce,8854642,1684,0.292284,876.850575,876,0.850575,877
1,dairy eggs,5077769,3447,0.167612,502.837344,502,0.837344,503
2,snacks,2703101,6260,0.089227,267.680576,267,0.680576,268
3,beverages,2513695,4362,0.082975,248.924226,248,0.924226,249
4,frozen,2080442,4007,0.068673,206.020386,206,0.020386,206
5,pantry,1749048,5368,0.057734,173.203360,173,0.203360,173
6,bakery,1101488,1515,0.036359,109.077294,109,0.077294,109
7,canned goods,993053,2091,0.032780,98.339277,98,0.339277,98
8,deli,982706,1322,0.032438,97.314643,97,0.314643,97
9,dry goods pasta,806130,1858,0.026610,79.828812,79,0.828812,80


,department,occ_before,share_before,occ_after,share_after,n_selected_products,share_diff_abs
0,produce,8854642,0.292284,8796817,0.406290,877,0.114007
1,dairy eggs,5077769,0.167612,4124581,0.190498,503,0.022886
2,snacks,2703101,0.089227,1361698,0.062891,268,0.026335
3,beverages,2513695,0.082975,1616072,0.074640,249,0.008335
4,frozen,2080442,0.068673,1133581,0.052356,206,0.016318
5,pantry,1749048,0.057734,865173,0.039959,173,0.017776
6,bakery,1101488,0.036359,644614,0.029772,109,0.006587
7,canned goods,993053,0.032780,588813,0.027195,98,0.005585
8,deli,982706,0.032438,656081,0.030302,97,0.002136
9,dry goods pasta,806130,0.026610,420816,0.019436,80,0.007174


,segment,pct_baskets_after_sample
0,heavy,0.713907
1,frequent,0.206320
2,rare,0.079773


In [38]:
# 3) Build sparse one-hot matrix
X_train_rules, product_columns = build_onehot_matrix(baskets_train_rules_apriori)

display(sparse_info_from_onehot(X_train_rules))

C:\Users\tcham\AppData\Local\Temp\ipykernel_12536\1627450262.py:433: FutureWarning: Allowing arbitrary scalar fill_value in SparseDtype is deprecated. In a future version, the fill_value must be a valid value for the SparseDtype.subtype.
  X_df = pd.DataFrame.sparse.from_spmatrix(X_sparse, columns=col_names)


,n_baskets,n_products,non_zero_values,total_cells,density,sparsity
0,150000,3000,1210930,450000000,0.002691,0.997309


In [39]:
# 4) Mine frequent itemsets with Apriori
# We need max_len=3 to allow rules like {A,B} -> C
min_support_value = 0.004
max_itemset_len = 3

frequent_itemsets = mine_frequent_itemsets_apriori(
    X_train_rules,
    min_support=min_support_value,
    max_len=max_itemset_len
)

frequent_itemsets = add_itemset_metadata(frequent_itemsets)

display(frequent_itemsets.head(20))
display(top_itemsets_by_length(frequent_itemsets, top_n=10))

,support,itemsets,itemset_length
0,0.165480,(24852),1
1,0.133247,(13176),1
2,0.093700,(21137),1
3,0.084573,(21903),1
4,0.075533,(47209),1
5,0.062533,(47766),1
6,0.052227,(47626),1
7,0.050413,(16797),1
8,0.049007,(26209),1
9,0.048460,(27845),1


,support,itemsets,itemset_length,rank_in_length
0,0.165480,(24852),1,1.0
1,0.133247,(13176),1,2.0
2,0.093700,(21137),1,3.0
3,0.084573,(21903),1,4.0
4,0.075533,(47209),1,5.0
5,0.062533,(47766),1,6.0
6,0.052227,(47626),1,7.0
7,0.050413,(16797),1,8.0
8,0.049007,(26209),1,9.0
9,0.048460,(27845),1,10.0


In [40]:
# 6) Build association rules
# Keep single-item consequent for recommendation (subset -> item)
min_confidence_value = 0.10

rules = build_association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=min_confidence_value
)

rules = add_rule_metadata(rules)
rules = keep_single_item_consequents(rules)
rules = sort_rules_for_recommendation(rules)

display(rules_summary(rules))
display(rules.head(30)[[
    "antecedents_str", "consequents_str",
    "support", "confidence", "lift",
    "antecedent_len", "consequent_len", "rule_len"
]])

,n_rules,avg_confidence,avg_lift,avg_support,max_rule_len
0,211,0.199648,2.153669,0.007111,3


,antecedents_str,consequents_str,support,confidence,lift,antecedent_len,consequent_len,rule_len
0,"27966, 47209",13176,0.004033,0.442251,3.319043,2,1,3
1,41787,24852,0.004953,0.383385,2.316805,1,1,2
2,28204,24852,0.012400,0.381852,2.307540,1,1,2
3,"21137, 47209",13176,0.005393,0.369406,2.772350,2,1,3
4,45066,24852,0.010007,0.358662,2.167403,1,1,2
5,9387,24852,0.004287,0.344957,2.084585,1,1,2
6,"21903, 47209",13176,0.004420,0.344058,2.582114,2,1,3
7,8174,13176,0.005107,0.334937,2.513658,1,1,2
8,8424,24852,0.004787,0.331793,2.005034,1,1,2
9,49683,24852,0.011287,0.328164,1.983106,1,1,2


In [41]:
rules = rules[rules["lift"] > 1].copy().reset_index(drop=True)

rules_2to1 = rules[
    (rules["antecedent_len"] == 2) &
    (rules["consequent_len"] == 1)
].copy().reset_index(drop=True)

display(rules_2to1.head(20))

# Optional save
rules_2to1_to_save = rules_2to1.copy()
rules_2to1_to_save["antecedents"] = rules_2to1_to_save["antecedents"].apply(lambda x: "|".join(sorted(list(x))))
rules_2to1_to_save["consequents"] = rules_2to1_to_save["consequents"].apply(lambda x: "|".join(sorted(list(x))))
rules_2to1_to_save.to_csv("../outputs/apriori_train_rules_rules_2to1.csv", index=False)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_len,consequent_len,rule_len,antecedents_str,consequents_str
0,"(27966, 47209)",(13176),0.009120,0.133247,0.004033,0.442251,3.319043,1.0,0.002818,1.554022,0.705139,0.029157,0.356508,0.236261,2,1,3,"27966, 47209",13176
1,"(21137, 47209)",(13176),0.014600,0.133247,0.005393,0.369406,2.772350,1.0,0.003448,1.374504,0.648767,0.037860,0.272465,0.204941,2,1,3,"21137, 47209",13176
2,"(21903, 47209)",(13176),0.012847,0.133247,0.004420,0.344058,2.582114,1.0,0.002708,1.321387,0.620694,0.031199,0.243220,0.188615,2,1,3,"21903, 47209",13176
3,"(27966, 13176)",(47209),0.014127,0.075533,0.004033,0.285512,3.779947,1.0,0.002966,1.293887,0.745984,0.047104,0.227135,0.169455,2,1,3,"13176, 27966",47209
4,"(21137, 13176)",(47209),0.021867,0.075533,0.005393,0.246646,3.265397,1.0,0.003742,1.227135,0.709268,0.058619,0.185094,0.159025,2,1,3,"13176, 21137",47209
5,"(21903, 13176)",(47209),0.017973,0.075533,0.004420,0.245920,3.255780,1.0,0.003062,1.225953,0.705535,0.049615,0.184308,0.152219,2,1,3,"13176, 21903",47209
6,"(13176, 47209)",(21137),0.021987,0.093700,0.005393,0.245300,2.617932,1.0,0.003333,1.200875,0.631913,0.048900,0.167274,0.151430,2,1,3,"13176, 47209",21137
7,"(13176, 47209)",(21903),0.021987,0.084573,0.004420,0.201031,2.377001,1.0,0.002561,1.145760,0.592325,0.043274,0.127217,0.126647,2,1,3,"13176, 47209",21903
8,"(13176, 47209)",(27966),0.021987,0.047267,0.004033,0.183445,3.881055,1.0,0.002994,1.166771,0.759027,0.061842,0.142934,0.134388,2,1,3,"13176, 47209",27966


In [42]:
# 7) Save outputs for evaluation notebook
save_itemsets_and_rules(frequent_itemsets, rules, prefix="apriori_train_rules")